# RepairScope-X | Notebook 16
## Full-Precision Decoder-Shift Ablation on ALoDLM-1.7B

**Purpose.** Notebook 15 changed two things relative to the failure-generation run: it introduced a suffix-repair intervention **and** switched the ALoDLM decoder from `left1` to `entropy`. This is a confound if we claim the repair effect is due only to selective reopening. Here we **hold the response, prompt, action, token budget, weights and precision fixed**, use `left1` for the entire repair grid, then compare it against the **embedded, already completed entropy condition** from Notebook 15.

- Inputs: all **16 frozen ALoDLM-origin failures**, embedded; no ZIP uploads.
- New GPU work: **16 × 4 actions = 64 FP32 trials** using `left1` decoding.
- Existing comparator: Notebook 15's **64 entropy trials** (embedded; *not rerun*).
- Model weights and inference code: same frozen Hugging Face revision and pinned public Git commit.
- Outcomes: task-aware accuracy, preservation, error recurrence, native loop-token passes, runtime, paired decoder-mode switches, and cost-effectiveness.

**Important:** This is an *exploratory decoder-policy ablation*, not an independent held-out test. `left1` is deterministic for this protocol; repeating it with arbitrary seeds would not provide independent evidence. Compute is measured and **not** assumed equal across actions.

**Hardware:** Colab T4 GPU. This notebook can be run at the same time as Notebook 17, which needs **CPU only** and writes to a different output directory.

Embedded evidence SHA-256: `ef2d25b2597c457339142dec4f8ed50bef4a21e6204bfb948ac76d20d70cb64a`

In [ ]:
!rm -rf /content/ALoDLM_N16
!git clone -q https://github.com/amazon-science/ALoDLM.git /content/ALoDLM_N16
!cd /content/ALoDLM_N16 && git checkout -q fe225a99e0b5aeff351804e871d7581155d7f27d
!pip -q install "transformers==4.56.1" "accelerate==1.10.1" "huggingface-hub>=0.34,<1.0" "safetensors>=0.6" "PyYAML>=6" scipy pandas
!pip -q install -e /content/ALoDLM_N16 --no-deps

In [ ]:
import os,gc,json,time,random,math,hashlib,platform,sys,subprocess,base64,zlib,zipfile,shutil
from pathlib import Path
from datetime import datetime,timezone
from collections import Counter
import numpy as np,pandas as pd,torch,transformers
from scipy.stats import binomtest
from huggingface_hub import snapshot_download
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
assert torch.cuda.is_available(),'Select a GPU runtime (Runtime > Change runtime type > T4).'
print('Python',platform.python_version(),'PyTorch',torch.__version__,'Transformers',transformers.__version__)
print('GPU',torch.cuda.get_device_name(0),'VRAM GiB',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
if torch.cuda.memory_allocated()/2**30>1:raise RuntimeError('Restart runtime to free previous models.')
EXPECTED_CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
repo=Path('/content/ALoDLM_N16')
actual=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
assert actual==EXPECTED_CODE_COMMIT
sys.path.insert(0,str(repo))
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
print('Source commit pinned:',actual)

In [ ]:
EMBEDDED_SHA256='ef2d25b2597c457339142dec4f8ed50bef4a21e6204bfb948ac76d20d70cb64a'
RAW=zlib.decompress(base64.b64decode("eNrtfQtv40aW7l8hfAOku2Nr6/0wbu5FBjtzM8DsIJj07gA7HnQoibKIlkWPKNnt6e3fsT9o/9g9p0hKJFWkqYfbUkIHcWKx6rCqKH5fnTqvzxeTMJ6tFlF6cf23zxe30TxahMs4mX+IxxfXF6NkNV+Ok8f59SicxcPs0jU1RF5T/V/hLBnP7j5M7jm7uLy4XyTDWXTXoSM0HofLMI2W5ZYfJjCKKQoKF8sY2+PVTe/sFnf32Onf0yhYTqNgvrobRos0oOoyoOQyCOfjgJkg+hSOlrOnIJmPoF0SQL/xapR1WYaL22gZcDEI/hItV4t5EAYPcJdxEC7i5fQuWsYjEHAPg0nhrk5kBP8+wsXglz/88c8//OnDD3/++a+//8t18L+z+/+fXwYwuEzwxTVIJpcXyWp5v8Kh/vLLL/dPy2kyv5n/r+A9DKDtZjFOJfgO5hJcwURu5veLeL58c3NR+fTi7c0cxN6AyHfv/pwso+t375zo/BEEi+gfqxhkBqs0nt8eYaV+TB6jh2hx6T4vjbc8rABuGs7SwFwG8yTr9n4KE4rn43gULmE0YXAPg50v43AWRItFsoBVA6nxBK+75/35Il0u4tHyA9wBemUfzlez2eUFTACajipXsqWuLnwuIF2NRjDIi+sJjCnadF9/vlysIuybftxq657Qh80XczPhbDBfjnKX7Mv8YZl8jObwqWWXxesXjTefqsuLx3A2+5BGo2Q+hk/UgGpDDJeKW/hRDCVFk7jchgyENUJyS6xroySMwUn8cB8t8nYX11QOJDfaWkasAKkCbnYXhTjhePlhHN0vpxfXbMCVkIavf0DWLEnusyF+uA/TFMGDas08F9z9snch+xxamkFFHmfubcFxLaN7EMX05cUiGq0WC/iurG+AK5EuQfwiClOHDbNofrt0gBGFHz88LMK7D7fDi2s9MArFG6okMyh9sZp/SOPbeQivewQdwygajuhICDnS0UTYMdd0OBQkijR0I0Mb2qGNeKiUpuFYjIjVbMjHw4hTMR5Rc4EjAWEwqovkI/yVfZnzL2oOqLsC6JfLneCXkT3hl5Hjwy88sYCpHFRkF1Bh5qXgFyS3wm+azFZLh7Xzh2T2sAbJdDVcLkIHLO6md6vZMr6f5dCE04Av3mhanUQ2rI5LMSiGUAX8N0wjfqq3wTvs8T38woYlyIVG4wQGirDqQNYt3w8TeGdgUFcRrN8KRjm/zfB5lMCbM1r6byPdbRTeRsFtDkVgpsoInK19V2zMP39pCG67zd4YTAxRWhMjEV9NAwQTLTnjFJtoIVUTBmujNZWSWilApPRisBDaUrX+acJgqztjsKZGyzryyt8E8tawc1fkNfsirzk+8lrYzekcbUgX4JWMvBTyoui9oLf7hCbJ4q4+m4776jcg7h3Ifgs7VjvY7KyrE6u0Kjbah4KkLWNkvkq/NpBUtg6SYqCoNEZZLgj+NGxUNew8pc6B1HLuR0kxsMJwzWHjyyVVTSjJhTZMEbipBAjifpQUvONOFZ4UILgCCDdGEOnQtYKX1PrwUvM6XkZJ6gVLogSzFNaGK3W6aGkOQUu+L1ryF0BLAeAicnCxXdDyitKXQksUfcg5gYDdHMVfzG7ArPzhIacEzQs1qCI5yHqIx9EYZICg8h4aFhS2yzHsYesLeiic5itXW8iz1fupD0412YZTTYkgiKkIlUJ64RS2kUoKwxycCmWa0BSwR1PBFOClgX2qH00b4FOIzvCpmJF4H/eb13ebxIueoiN6Sgtjx2MLqujpoic/CD3lnuApX0DHB71W7HJuKF/shFUedsLKBGzuYA6AkhvkLH94CHI2L9OgrvuXbpnBpztfyARVex6spVc2oOLM959+wJTbgGkZI5xYYv07T8MJYBm1bndqRZN6zrhWlElFiIJtmm04IvVjpSSkM1aSOjxSLzzqrptLxmEHrCkDwjhZdJSHgKPaExzVC9ifyKWzrOBrT2kXdKTCvhQ8ouiDLFBOBXaWHVoAZPmzQ6xPLesU41QbRrU+fXXL9r50yLk5U0ivsyH9IZ6Hs+CHefoYLXBoAQy3PPyD1fnq9jNf7TPGU+499GR1PJWgDIOCro1Wzqok/KiqJG4qrUNVzngjrFoNW1VtjbBSGuXX5zmVjGpuKSA1qPa64dTTi77+U8/8ZCAXKrtZnrrq85oYQEmLlHO6kKsOgdx9lfkX0OVRP10bsnknyH0xwD0Ibt84hZ2St8G/wEQaDyarrY50Msl4Bcp+jRtDsQVkfECJIZaB3th2MGkVA8WVOazTtMGCzgeSckqZoVRIwEc/jG10YPztRTGju2vVMHjY2uW/af1MkvkwDI8LOmEYF1RyqiWXghBxqiD2vFJ9m96Zj9fLRRjPQeMRrA246m0rYOUudoOoPyzC+ShOvw0m4V0MWDQNU1TtPq5m0SxKLwMR3K5ieNnSDLXCYBQtlnDTQudLJsFDnMzieToIfo+7nbyrEyQCfGXmt8W1TJS7pGqXMiH1Xn+cZGCYLGF7tLljfj0I75J8s7YZcGW4+dDc2SMpul0G0+QxuAvnT+vrzlS9vRQP0f8tW+krkPm+ZVwTWN31kPDmNzdvGPxaxncRzu/7wNzcvM2G6Nrmg86binVTZ/MW0HYQ/AnYAK49wB/BsEwrlUfg1PV/rMJ8e4k9DGrsqLaLB5AmiJP2czJ7wI3uJFkUUt1W9Obmbzdzzjat4Sr8zN1fJv8D/v9QEDcVE/x5YzhltqsJ3ihOmIYdJGI49ev4RoBATpVrI9B+4wdxi15QsC2ELSnsD/06PrWdTPA7bEZJ2Qtqyw2K0wON8ZbBzlpqobQCktKviuWtcPwsfkumO+M3tN0Xv/8aztClZhLNZhmUOG/H+HYK+8D7Wbh01meEpuQuCtJROJkkszG++Q57sBc8pWXwr+ED7vecew58y2fYQgSw9YNtq4NhkJrdAwADVd7x2A0M4G85XURRcJcsAJQcaC2n4TJwX0EAqgl+Po2Xy0Ktvl3A6znOsA//LgYJaA4to4doDtCb3oejaBzAdwtV7Kmbz2XwiILXk3oEqsiGnczLMK22YLpQ8tcm/GLR8ok2rhuOEb/E2Kh9hX50iDzP1si3PltL84ayvPfb5mXKAL1liaBBAgJBzF08j7JBfpuuezhMBx3i3bv3jqrGMXz5UdvAGUTzd++u13MI3my+COhJsB5e8KY0Gez3FqiAqmLqN3OG8n8q1gzHBZNA0TllBdOgRFrZw4S/lo8RLBjAGWDaahk/lOZ53MNjdeb0QnhXehEENsBMGKcjMNJgbgP+YYpkigTTttF5wRLFJAFpXHLtd7MVyjlk+UjFENHdr6tMUmgAPLKHl0Gjm2KWvLqHVytFPMspWnfnFGi7t04AGI+var4PXkSrObS4DH7CNx53lcPVE+IUgMA9olSKBDNM5tEVgPVoGo/guTrsnIazSYZ5sFOHt8aBeXg3XMGLiagJsAlgdZs40EtQLOgejkk2cqcJIGZy67DT1xvemaQqIkUfVMCqNB4XkPsI70DwiMhTjCkqhlS6BYosNIPNACZJMs4EQCeQX+YaXj21rmDxRnGpycqW0UkEcUBrUQAPYrQC9Isye1q8jO5gao8ZVOJrEAyf3H83gP676noj3q6fRwHKP65XCy//iDNPssnnvS5hX+Ag+uZmsghHn6n68pl9ybf6bzfSuJOWLxPKYsWkMt6vPpq1VKd6IFuQqjiB4n7Gx3Mo0FeDKey5KxKkK9JLjH5QDu8J4X6gV0xLZqXNXNnwcNwP9Fpoo6RRmhGDTgk+PYJ2CqegpiPkAybbssAtPYIfqkcQDatjOQcWg+nJ0wH9OoY/B/qaWtsV9LHtIaCPwPkUDOOPEWDrfIFQOpvheUmEqH8fwgb4G1N+5Tef4/E1DGHp0AUlOK0hkwIvZxp8I7fb5CcxK4A8dwKD+17YMxYC02wzPc57Gdcn/b8Vs6FoBuC17NIo2kU7LJ7EC9gHbxAZQxmicDHPTnZwlVxIBbzPg2CNxj+hGOQgQGUnywFuefYFJv8cZbpDtaF/fe9zqQ5QYQG/w8vfB9+A9o93/nP0aVmjjw3pVIaczw/Y46o82PxjHAJI/5//Dpx0QQ7efwtSNTeeu/uG7IbLesCkVFxl5zsNh/SSS8sYFcy1EU3HO4xTYqkVQnPYhDe4cOjGwLbuJzpbBkV76HZbCzSCKmrEa9sUW4H0WeTllHZGXmi7L/L+EKQreDLTaJHAmzxHv6r8kxVsRUGhjiKHWbCOuJOCPTkeKs9Rc4/n8OTyY1/X5QG+bGF+jg59fw9/X/1bMoQu+dnDIsYDEND9cjnQB4BytRhstr7ZhQkMHIbkRrQsxLsx4jgyxwfsmCHVsj4A7OZuVoZrSbbOZtZiYb+bzTQ7tV7vSHHruASg+5yN68uXz6L0iZv/F9yxsoGstnRTy1p8Cdyh9x83g36jSNH77eX2/LIh5CKzk3iC+1hZH00meGv582VGIbjU2+PCQZQ6j+PJJIL3bRQV06fyYJcPWfGgI+d+CiK77o0VIxQwmInMDNqwN4bNNhFKZSDc6HMsCWWUKK41AzBrQOHqTyMk8+6xxsoT53ZwhDGwjgR1ADagp3QKsoWyPlhOH8K7+7zDCDT4K05aT0J87Svw7Bp0g2d4P2E/hievgEXFi7oMhknyMXVbSFX69C55iDN0Rkz5Nhgtwn/CNhq+ffB7NE2SGXyYwlsQZcfqT8kKvcTGGAGRq+a5CJT8GC5HU4BEaouL2V03UI1bxewzB8N5X7RcrkVXALjxbBwfnTst2X+y23PdTBVv9Y9VlGahyikIm3aZwxCPiHEStXQOxajd4c8dpmdI5vlRDyxkaQVh5TLRdXG4fwZtIUWDgAtXdhvpeU61eCodj6OF1/q8WYz8RpnE941N3QNGJ0W7MXvUW5ZEQdODsb8avnfmu2/TFfk5Z5IKazL3aN6w/TZKccoc8lPNmnz9qLQGtupaGp5ht+/4G0/IG+DeyO47cMu34V4dvAsH3QH1DKte25n6WfTuBPlC7YT4Qu0L+P8J+n0IivsY1HEug/tVOr1a3WdvPAdEX80BlB3s3T7dBaMZPBt4ncfh0yA3EmJHmevx697D1TIwOoANYLqR4QDvn9n9BjVYX/d0jdZdUPh6hEnlMJo9cxbynOjSxL0nIWvrYnlFit6DoNu6tS6SG08uxx164OHLRtAiKjmp4HGIxAMLy/Gjv+OVjWj/UtdFr69WRQdXgTmqhws/dxcXQruegBCrNMCZkg5fG45AJKIJ485R0VDT6KfINBdMCmmMgLZeDAYgFx1cXCTpDMeMlc2RdWskJwcDs5SGoEu6NAr9P08JmYXaHZitFDshM7TfF5qZBTi6u0evhEfcgS6Sx8KXRanqpWn8ES/lm1OAGTwNHWAaGZ+EvJlz75gnybwOxi19ctEZwNTlVPC5cfO9jj6ZYOIxkDU9ys2zPelt/BDN1y7ouLUvraJX2mV5Mb3rmLuwqwY5pREcsM9eT3uatE0dR1IEcdfcIp+RUFuoj9HTcVML8TPfd+vOfieUU0W1yPbUogHzMQeGUCLzTdeNbifKMiqFUpZoIhpibCrGQ/4Sh+CHb78VM0Jay4GekLpOCeXriN0N5hXfDeYV3xfmszRhkcsPFoTBNIIugIWR86aLnrJ32VnxMovX71YjBKlpeBfch7Mwj1teRJfBX2DDF81gKxgu5gg5eEIgZPAQp/EyAWQAhHChN14pQTLPTxuihzhZwd4b9/h/nAQGEPQ+Su5nUSYpGjcIwHCcHGycd52RKKN0gLMeSLscNxA8vg8r7ifaiO6cUvcYz2+8Pfmf1vcM8Uh+hHCId86ZZAPVaxkJgOZ6bsUJh+de3vXMode5t6SFjyY82swXcnWPZ/Zrl/ZNeGZtCMVdszGW7/Btum61NtgWnomfggZ3+uqAy6uQ2XHfP7MIcZobFe6ek7dxhC2N+WDlo3L2n39Nzlj9oKS7C6QyVpnseAf92n1UZKmQxKEzqB9NiUaBigw3klklpNbE7wDZ6Pxouod38jaPeiYO9oQhhgDpAvG+vifMs6Ty5e/Qfb5cJPdPMLfcJubSPMO3KpztmKD0v5J7WPwZIO8Hhva0PdOcHjFPdJi/oBe1kSWL+BZjwGHS6T26RFe+5PiORYsH+Oq7b/OnTQYJFzjnDtbhYrqa4MXhapyF1YjyS5NfLHrixUmyGMGVyaIYFeYOO0q+oBRo9cPjIpnffsi7Fc3PNHfy+vn0ObK75cheRPchDHXcL1jHBXMDKN7v9Qu//t5mNJKkbudcXM/fIrqVh4V6D6m4z7mT1rzvlT/Wl+uuaVrlQFeCxESdpIE1qQH9yBjMBm649vtA2YFkgKlA0tCSUjyI6xxEZgiTTEg8D/QFBI8JNVzRaMJCRdlIGC65kNxENgSin9DxeEgjS8LJMBSg0A5DxtRkjA4AZiTGRrYxHupSB3KVJKfKVW5ke3IVOlk0cpUwLVzFjZereqrqqeq3TlVbCdp+Kd/1l/YcbU4gdFzNls6cCet1GBGRgaoGCohO+cEwH4SHl6zQXBnBKJFMcdqQrozLjoZ2LgdKarOWWqclM6CMoZuXVVkiSi8riYHShnEuCWOYb8J0TFJhYaCaaQN9QaeUZ8dJ+mT1J32A/pRp1w2cpFkLJ0nq5SQmj3MA4mOlggIOMlCY/WnpwHOhnpdOnpd+BpS+Gj7Bmrlz0vIMvodRZ40qWeqqCZ/KPQ5lEoDJsl2/nsacs6a0Z55TOkKtItoITrlmsuHMTpmO5iLOB0QikEtmBaWoGNWzzikCCg4qIK6UEW3Qb4hlGrOoY7ZOomnXjG2CKUkth38YsWdCJLezZBjOTotC1mPakzxIC3dgx2buUF7uIL0+0/PGi/DGc2uUzanjPLgo8pbe3AyTT9H4czHkPDM0/HlV4Y0vh3IBYaTdSiK88cLOu9dDBw1Rc8x2dRiQfFBOjkfFliHKDqxm3AISuzQb1m9jklhuRBIO+K+YoNx0pAChtZYWQJ4pfboUUC1ZtK8t5uWKxv2abDGvoUtwa16tkFoHUuiL5tWK5nXQJ/o1y9eMY9DJgaxVdVLbPgvzlxIhu1CWK6XahbEwIL8yGuUpfaUZ5q6WWXJC3cBYlCuD9QckBfITdAejjBJEcaAiKsTpGmUaSWsno8xXJa3XMcqIg40yr8FZtGesnrG+7prRzeJhCssY83tlC5avhhO035IVdwH162C6ki6WsXrI5vWCpujh7fOFq2ppTa5xnY/Z7KASiKPEdrAR1Zii0JIskMgfy6kGCg/jLBdCM8GU3sF1TqGtSBJ90jabRsrSJ6tnvY7NBi+eoc2m56yes3rO6spZ0uswTY0/kkhWf/zlIzntmthQ8AEnJXlbWhalA0a4kpYaV2OdNlCWK3MJbYwW1mAUbedUt4oxIxUVmANHnwll7Wwd+gpk9WrWIXuwdehVHAt6kjprkjpsmTxVk3ddqKDignCF/w1+/+l+Fs6z1xd9Eugg+BnELzeSZ3izdJnfAaTrLD7r50pAcPRpud1SXQe/5I8BGe+XLM3xv2Vze3IdM984zL2c/XUXxi7keC1Dggy6zhHHDrZz1VQo1L+q5OZNAkatPzhJdcnay0VXckMHukquhS2FjDJMW4lJ44U7RDTcS29owQJi09oqzqjervPUElWL9eoI4xLDmszJ0ps5iuVr38p1zBzd8qVsC9VJ2sJ1tO0QkfoPETflEzGQz6iXo74jeEZUU20xch7UV4ZlexlQnaMy2bL7u6oe5RrZMMWuZQKZq2GqsZ6IHTSWCSy3KlUJPECTesXZVT4te5Ws/eYOZQktuQKYNUYQqa2uF9Pzpu9l0utirWjlh/n9ImTX0B8xYLr8Y7dDf5iVxBqFidmtFQ1WJiQIBgyCYULaCEt30H+YtAr/AQ6y7OwIYjcr09ckiGesTK0EwduO7LhsC/2RXoKAR1t+C8yBiJu9di+lGh2dIHLSauGHizcWz1e02/iTwUXPFa89u8rzwE5l3Qb/Bt3mh/E4sFnaUn0N//fdwdE4pFI7mnZKKUyJ32vOmA4lYam0Hcs+CTmQFR86sZ3MwVItDZVZQI5pcEgg0kJ3jCYFHFd8F11CGU4VU+aUHakbqUKfrC6hD9AlqG6hCtkakdNg3RGKw4Om+M1kBIsbnDBVUP3aXNFTxeGzy4ojeIe/nmLDgV35URSWl9wJvFig6iGhC/AsSCNYPsajqGaxAUyFHuOD43EqKnk9HEd4w3HgnfN7s8GuTVGA0uw396sdAr3LOlEJMIVgnGhls99b51JAhcAyoE1IlxqaNsV2AscZi7VLCGD5DsneFJAQh83oacfk1Khkd6vLy5NIB6tLK320WV2UbXNno2dpdSH94dPXYoluQTTt0yliaGAa9SAauzYjAI6vLQrk8NgZKqrasuqU6pISv5OXkgZEFb9Eg48X66oEiIEw2pD1L09FLUYEwSp+RTUXfxyNhu0/1lWH/TyjXeNoOGeYEoAKwuTpWhP4UawJfF/w5se3JqCTe7M1gR3bmlBRMw/WAF4a0ivHRVeUnlWIvsDQP/zF7AZxyx8eEmgpAFtFjq3WkwZlTTcg6yEeR+Nsd1w2X/tt1bjKRwm4P/HplzQDN5FSFGo8OZRots+HqinJvFW8pY9lML6+g1uW5B3tEvBKbYWP1oNfuABdhBvQHzCDtWzQD6jgRBGrLZOCMrOTWQK6CEFQP2FnxzQ7mSW+KtM8Y5ZoZRouj2yWOHFH4ivaU8tvhFpa3ajymsV/HGPigEnm3PSXbG2wllqK6/bvadRxgcp6zRU92KO3ThvGSxsNtWD8QZK8Y+0XpgZVCuPbPMEYN9pyzOlvVYP7rhkwdMOlUnBOJDO7HCQZV0mBS4D/03Xf5UexSXxVntAHaCTtNgneppHYBpsEq26WepWk542vwRvVs6/KpezY6zATQR27vdkeOfe6IpVTMwJo+s3L0nYNHqSDcu5IabY9kZTgmNdfYKEyJhuqAOgBNZpxrNbOsYC47nqyRIjgoCZoJtTpGgT4gQaBr4DgHQwCrdjdZhDQZPekwyduEBA9Up8MUnczGHTdWNO6wQBa5ysjjmEk2NqRVI0EXvsumoF9NgIA3BKaswZPIcE75y5ROYw7qWI736KhmkqWOZUqpRqC6iRn6CwonT3BiK5Qzrh1pWKokJKeKpTLY9gI5J5wLo9vIZBtFgLVZiGwu+/HtVC1Cgjnk8NXnlcOXyaKNIFsg/HlDw/BeHYJ8rvlSS/dsj1J+nF24a867QZqAwwFMYtoIygvVVdOJ78e48FRC6byjtWt0NSbwVGo/QlG8e75fKuZ5mWdYPhAWcMEZcS4osbNubE0qBRMA2cYt+3veuxjpWLaaikUN6d77COPYR34ihzznG2gjWN425kP36NaibAVExSXJx3UxnqK+e1QDABjDPAwX37rqoWGM5gexqoFj2Ea3OEl+GbDWKP5aJakQB85T4yScRQMZ8noYy2dceHcuiaedJqsZmMstOleuDlWOgXZ97MQRUWfliAgWkTfpuXu0AYkplmAxM28wxL8cnAEtiT1nWA1IZY/YoJrrxWbVXnK7+VKRPekwyWK2o6tMwNLNKOKuqzzWjX5uDKlGAGaAWUGOWWHeAn0bqUCCEecsJOrPIZp4iuSlD5AEaJt2RulblOEZINhomelnpVemJVKH709it2hnlnXdC5+iBhdV0t8ZgfdMc0uN6ADlH+2zQ7aGGsZk5gXniOMejUJrg2oEBqaYckS1jUvvMLwA6UpYeRcjqp2Njq8ODZ3MTnIfU0Ocvc88KTH5B6T2zC5m6GhsQpg3a5gcDDO4nK4XaGGrrqbWcHskgOdStLRv0e5eu1btgOhqDWGuGK0yjZ5fsK+G+3IhsDem/GOgEyVwNJTLuM5P1U8VscwHag9UVkdv0gHa8FmZduKdLS58uDFr1yk4wjoTCsun1TY8yr452KzXHUhWgB0+bNDijaRS1fwyAXO0s7xu5ucgLiWlUjeTbBbhqfv3v3BnaVU6uX9Uh7+L8cp8ncOi7RtgsDjGjfM9WoU0QlNDHiwBcKuE4q5pBh1C4Q3cZLyJ8PYJNVwMdF+byXTWW1w9WEr0dV1v1NpLCeMK+Xy6ommXBiUKilAA5DwX2t3KM7BLKgZcAMCuoM8N6bayQDxFZnqucocbUwl2ozcoi2BrFBnWZmDqp6qeqp6gUXy1Eh39yymlhEOxd6+xBnP2Tbi9OA0TaKS7dLWlSTjD9D2V4qqimoqmm47+l5xMmAlyqRaebI0Sc4otc7sYKxoKsJBpRIa6EsBj6nuGc3PpHC6OobZ4StSkz5AiWJ83ywbyp5n1fSemXpm2j+2Lvvm4OD/isnIQfkpL8COZObSm797h/dNFvCUEpgDPKcAXu0MElz4XilNei1be5gG8TIYwbOBxzr6eLhaJUhZEzJsy2bu1atg4l76qmhoVDekhmK2u29XPVNovVCv4fgPY1hDStOGKBBCOUxOuIq7nP7qPLvUYSaZF+etLsU42L7FONqSCnJ1hiaZ/tDv7PiqY4X2tvHXarKTTT6p7MZHqMJOa4n564qKN8WH5V6g59Xc4349RTHTOZssq/xsqymccmO1IRpdeAlvOEHThGqitSBKC6G6Gt65xsK5khNuTzhG5Ch5pPYN+Tt+FinRhvi8tSYFbXONog01KQyxkhW/6Wkb5XmFAc4K/9+4mDZK3gb/EjDemLev2mqn9K4vO4ADMNY9qCqkeqMivAHUmlX+8e+cRdfEfDiSKoKSgTWYt5VwtEEw1VBuFTYflAmAUKmIEnSHkx5DtZIS9v4SeIaLc0PRnYwQXxFFnzFCtKIoa0NR1oaijH5tI0QPm6cJm/5bVo9SDnbb19UA5rpp1wuj3O9mVAuE9p+gu9oynXIK8UE1tFpvu4Si9VVQQanLTU10YyYKIQWXFjan28HLzajKhZWw61VCa3J+e1N9ontTfcDelLYdR/DW44iGGgfVvSkxPcr2KPvVUdbvzKl1gwMNa3DnpKqjOydXznBU9+eEzScTnEqCySCkbTBHUqKkUbDXNFa5MmSd4ZRKgSANW+yzgdOdD3RfHEg7HOiKfQ90RRuC0rP0se8RczOA9Qln1gP91PnhZ5uddoyiwTG9y46xc058aQZVeWI7HQFs6YSCPah0QURNkZ4EAArgTWnYXjqs6uazTgmHDalkrw9wt+md+QgPMIzn11II1vn8sq1jHc7qbSsQ5i7u54beFiKkW/VqsXuMkK5msDjtirmV/DXsVdEMAWyZLMNZ4WucTAIczPw2DSaL5C5YfVzNolmUIpTd3Lxha6uKCL4PzM3N28wa49rermKYVdFUrJu6yusC2g6CP0VLvPYAf2A0/MY4hTd+iJNZPE9zf7V/rMLcsoQ98qCc7wLxANIEcdJ+TmYPaBeCL0QhNTcC/e1mztmmNVyFn7n7y+R/wP83mqJ+00ty1ANcr+8e8/MIbdgSC9nR24Ha7WNbNoCNmTGwPRPOP882pCx2GYeJBpTG0rvdN8QW8wKgxUwZo9lZ8MX2Se3X4IvnnMHb+EK0ZaMRbdloRFM2miphiJ4wesL4LRPGNTaAFXCTv8IWNzfx3f0sxmEWq1B8cDBP+DZsXc5SmnzmLC0F2TacqziTWqcjaz2oCjSeoFnQLKiQVLiEm01Z8LGsFreGWEpZ5uPbjVI0Vdpozg2TyohzoBT9KiqIPkAFaXXi1m1HKLqp0O4mOVGeprWnlJ5Seko5FUrx+2ETf4pNUf1piCMyqqsZFChDVf396oUehQI2yYo9Wsv8aoodKDz6AoXIGMP1LpwiKdXEMspPnFP8Z/UvxyZdXKv3zXZj27LdWHmWCfZJTxwnTBx/WITzUZx+G0zCu3j2FEzDFHPq5OtwCRPP55nNHO4bfApwPuvh/x79vPMOrrso1vQyS0qQCXCX1OYSSnOXM0HVntmiND0qrHxABkGR47K0dtf5apQfHCxL6dl854a/eaiwttDl7zfznx2xTJ6Qh5yUgy0kte0dr/MLp35+MV5vRWuqJxB+frFEduQXGGB5cHqbX7ghMBirXZEA6y/3QtnAaCYwOyboLlwos0MKBbQ7w2SIBr2HytNhGMn0foaTWsdWnoG2xzKcqDbDCTu24eSM8jCrV2ebIsZmXc/9r+EMIADQdjZzWI5M4lLIx7fT5RQT/C5dCXeER3x5sdG/hg9oMJ6UO4rgLoK/U8wDjALmwSSazVxEznjsACksktVnYLechsvAoUnwhrK8N9LSBKMip/FyWQT/3C5gnuMMgovxpEEIraIHl9Y4vQ9HGOb/PgGBIOYunkfZIL9N1z2u16Gd7x2Mj13G5FHkZhDN3727Xs8heAMzXWYSsKz8enjBm9JksN9bQGyqiqnnkZ0/FWuG44JJoOiceYNpUOLeqVtk+Gv5GMGCjfAtGq2W8UNpns0J4/pH+TqPcrMTWN+3eIJNq72Z+hGSTRi9VWJTeUtsSr/flr/WQmc9UA+kea4UszKMCyPQGxYYtSFQS1jcOuDmAf5Ld1AEhbEUa7+BaGXtOdD0M/aqF6Lp5+xVbTQtzJHtVaeevKin6R7be5r+2o/SHVS4J4cPCNe8+oz+vNa14/kSSXCW5t3KrbNgboQ3l5gDrs7RsxAbrLsFbx6nEUwAL7oLhyd9IoK21EPi3npItMFzxGzVlvGWmmAd/UgEGzBSlqm2i01gST50TuRI06qpIhKo44QgNxLBpFA7aNOaWa7RrURioOHpk7R+FV1aH6BLo0Z8XAtgr0v3JN2T9K/sUT7/sKBJELg1xaPpSzyQjj4tP999yR5Y9SP3lKofoTkA7+Ie2Y/ZkwAaWQ8L7nFwkipjK3pnnW3FLmwr6qZMH9lS2jF1CbOD6k7AbJMtUJsShlnRUtnJDCTVhDMDVGcp7R4hCtxMGedEMS0EFadLth1Mo0el2S6mUbWvabQtgy9l52galT2/9vx6dH7NngBWIkyyE9JJPCk9rUE56ZU8ehyY12eGq13qk7g8WZ0Cv0AprPwIT/UoJSQ1Urlcu8gU3sAvLoimVjFpGWWia6FzJjQwgSanFvgFWuB+9stax1YygLa9/fL4OpcocwJ/3USEFcxcrs0w9wi5KR4STZJkHPyETYLHGBB9uHq6DB6jYBTORivAmyjz8IiX0R2g7WMGaQgUwfDJ/XcDvL9L5tFVDEA3jUcfM8gF8MxuVYDnj+HdcAUrs3CHUj+GswkOwuUYz3pdBqk7joJ/8ZF/purLZ/Yl94J5u5HGnbQEOCa5dbJYMak7JBigH/RHKW62luq8chDVSVWcQHE/x+Mo9Ss8/TLuvoxHjSnzHg1qr5tN7WCwwc2Gd7Teye36WXQgQHGQAtQWNNcR2uBYo10tMAq8IBTDKiXd1ROsV4uJwaQ1VAh5DsT0jMXuhYjpt2Kx63mpB9Sel06Pl6Q/0bufmJqCnXlXMlKDSpCa0tvUZDCBsOLcaUtc+41UTA0E01JSSiy1mu9QmkRqhrwHnRWV7ByISb+KxvSbsVIdxZeE9NzUc9NRlzETDDLTZ1c0yppNcWVxoOEdfmFwiMOq3Ou9VrmQz7JlzhegttrXnqU+9GCPbZFF1QfSm6dDKX/sQtWFsUGp8gc1+LJ26IEUrfHWAqhOWiye5ZLXS3+NLSoGCiuUSIpB18rwHdwtOEbeKcuwRgrW+DpVJutgAjoqh72aCcgcbALKX4sKFWSfvRRzZVy1H3M1j7bnrdfjrWyB4hSXxxUgUaUlrBHCZUEa4bJgjbg8luCNyTu/vazhPwib5lOqzyDA2i/rbmiaS3G02W3cY/Fx1VraG1l0HgQ/jMe5nS3F7wN8SafRYhNy6Mqk4DrBDb8LJCwWt3kw3cbCxO3hJiZa8/zbcv1jfmcEv9WJCyrU5pffGYGzjjlVmQGmxNz++S/t8UWgQBEaGBBzD5qmOG0trGIarjOBZVG6maCQhLRwLv2gip2Ob76Gwexlgqp3bCMjbNuboF4iZLtaDUucjkI1TR6DuxXQzI/RfPEUwADTACDYuSGnmU8BuiLEAGomGMYfMYL5ES3gi3RZYqppjPXSF/Ms8DtZZNSFYy55b/2EYhD/ALWdrGw7vr7vN7Lgqp+jzM+h2hBZwI3tG1OimvtcqiOabyQAJ1z+PviGcmeY/3P0aVmj1Q0ZV4aczw/g+Ko82PxjHAJI/5//Dpx0QQbPq1n94u62uOhxh39vVvlQpquqXFs8R73nhdJPc8RLbNp0zGnF2XO+FWIgGDfaSEoxp5VQTXFnoF8RrjUVWmrLd1CxBLHKUKBiaok4B25rt2K9FLf1Vqw9k5H01NZT269kcQvPRd86gtjmlTS4kN/IfCWzdj8fnFirEgvFRbeyl9TPY411LyTvmOuE04GtJGg028UuhVAaBErjiho3VRTCRFrEAB9pyrToXtXYUi4108YAY0omz4HL9KvoaX14Vk9mPZmdBZlh3w6rmzf7GlxWXcicy7JRHRwOVU3wW7eGceP35BD+tPW2Q6pIJjqramLby9AO0MEQSEKT5qJOGmOSJTFCWcqk2S540lzUyTJGXRgz5QZX41T57Hnz13GZ7NXMX3jx7Mo09Qz2m2GwKmfgKmaLV8zErSeMbb2+6wtv0hhDsSozeDsI/rAZ2WWA0V8oLi0+ygxfIGf9STml5H34dBehuS3dzH4chDjd2Sx5dGkk5+uJ5q2vs8ScRfJIWSTmDMbQCROBwgMtno+/C8wODYqi1ul9eUxZjg4SfHew1ayakMJ5CVY4S3nTTzLmV8MEqyh1Da6IWtCOnGUGVrnMWjUXRIAATaS1yposnVVDOULgNs2Bt5RmAgR1Zy5pBckz7EtxQhYzTul+FrNax1b+gra9xexFXBArTCbJqwfzpit4X6bRIvk2DdL7KBrnmYXX7hKkyCcQ3MWzKP3y5bMofTIHpIDPMJXxQFZbOrtH1iLLRBD8cY6xvNNktQjeKFL0fnvpCGU9kADetGwIucgsIy9BTJT10WSC1xN5iAEt43kwXsQYmIxCYMk948JBlDqP48kkAnwbRcX0qWxMSvGrXjJcruoyNa8QcXUiycGngLbVa8ObRLGJfaoOhA3sozrGDVM5qAZ5eeKGGZFGU2HRf5DThgwSlAwM0JTkUmDaJb1TUkUC/wphqKEnlPu4hYaeMW69EA31SRV7FupZqGehl2Ihf74K5vccbArBUqKjtyDo3s8yj1EuhR+eNVnTFB2sYEssLZPMAikIbXbyqjAEHeNPzBLVTDz6VfSfs7VEfX03dnkA7Rzbjb0nnTMgncQdsK4W7uDuVdkF7uJucrB9qHo8xrtVp2QNlV4q2fJYQwoK2dH7gZqBqVic+HZ1SsaEFZxgfncgHb/3A3AX5qaVoIsB8fBdlB1iOWhRTMMdyClzTgdr0VHZ5oytRT3L9CzTs8xpsQwn/oJi3O9YVw2qanCys1hJvWNBMZblqKi71llqNAfsZ0gugssmdjEak7YyqowWQnfPKmGM0ZYRrq16dWeE9CG8u8/xfzQNZ1ecdM/F91znOtn42lcIxzXYz7zTVsFS02Obd7YKsZ7yMZuq8NDJpG3Fr6iL88SyC5gBFeC0wKRlMEySj5lVXJU+vUsesJZunCUK/Xa0CP/5FKSAd/B7NE2S2bcBfAXiwnruHjlWWoQJpJmLRDh/ynwOshs4z4Nc6jR8wFJN0TyAd3g8CH5MHiNY+Iy1ilHD5JMADd8gNg9SpaKQgMOlNhddF+fKPyZBirlqoWOcOWDMkVaWiUuYGo8BoJeeQpKbxchvlEl839gUryOqU3u5LjBVb1kSlVW7aE993j+uvR9X25PCh0SefUjQSg0OPlnU1ZKc9Wgtr84n/FXCah7zfjLWVHTOkL7tWgHqGwNsFdKiYwVt9KwQmBID+BRUPMLIDj7uXFkC2iXGOItXjtfagYa3zVpfk4afM2+10TDuzI5q3hLVCrNZRvyTdR6ktufhnod7Hn7mcT3ivCpPCr7O1WdVeUZAd2k8jGfx8imTu6yUBlhG83H2ZGEyO87FLRg+rctgnix37zyOH+LxCkjt6VDqllvFtCvULfzVTTTxcDcf1JTyhlpiknSsJSafrSVG6YBaa5ybpKNy3pDWnhmpjZGMMyaU3OHAFg+CiZbKmlc3Eu5A5fpVNWp9gEbN+LENhrXi8+KkT3b3JvIXONftabyn8eM/rsvgNn5wFat3Yeo6T1ciNUrnzeV62N61LhbZu7zXRT4vuz4Hd12/4OF16QDe9cDjd3nwOXYBSzkFbyX00v5wOr9zaDVPScNBNhNadg6n49oTUMek1gxoHAPqbFNAnYY2wJxWUcqF3eEM22K5MqG0kpSf+hm230z68lzbxVwq9zWXyjZzqTxDc2lPqj2p9qTak+paqfW6ulJMSOk7kK4eBeqG0gOdU7DIQV0fqem0bEA0JRRuK/B8WvMGhjUaGJghyRohdkgnZjTm50T53JyclVio/Y3E1b7PsqxQxzIR92XbNldMJZEzO7E49jWi3q/S6dXqPgfVEbwCoykA1n+G8NVYPAH6jb1R7Ot6mUV3hL6i96DcPeCy2oqX2mWVKbGVNL7x5HJcwDoGzm8ELaJN2mGOgeASA8EtX6cb3og2OpjBEm0m5xW9vloVDeBsOiZg+Y2t6SQCJj1oUQ/LjFktRy26FNLxE1utvltjbrGuNXU8uiIdCEuZ0sJVHbWC86YgDoJGU1T7iCA7xHAA/UlOmTFUKHsuVNbB0PpiVNZHEe5AZbrnsp7LTn1Nt28JzW9u3lRWFN14d2KxQmp5SZ1UXEwNQq8Otjy2lzKg3lKlmF7MFyiylZTMF5uIZ5Wd4kTEQCrmcRuiQDbWEEFdKHxDcIgcEEYVwayYwNVE7hQdIgyRmgJfGqbOhM30KypmfXW4ns56Ouvp7PTpjHlzjHn9aODFtJqVfppKbxPdOe6RSqbXP74iccoCY4AGhlGP1jQViRPGKGWU4FbTXU4cjcEZGYZ1DDA3zQkTW0eT3pEp7TeULfMo+V10z2W/bi47whQrlPHW9dwaa4kAjNiO5CwsX97n4FKCFuOEKcLtrgJMbAlfR5eU043tMoiX8CQt3ugN3gQDLeHre3BWy7qniNyl9BvjlVPEBk8RpjsW1Ga85ntitou/McEJsQTVImtpA8VQrEPKOBNac6AjKXbgGHQ+xQxk5vSsWjCR/c1atc7Pcg207w1bxzdssUq0Iz+ZKAt8BoCdAEfoFz6/rTHQCB4+FsN8RJeERfKILXKXBLg+z8paAmDnn4UTwJR5kswzLMzcCjK7frZpZnYt0ivtMlBq3WIaf/S1QBRlqkFOaQQHOI2spz1N2qbukiOvhkv8GtZcGJ6RUFuoj9HTs04k/ZN6gSfV+fHAldSV4o0zz6VDCRjUoDZDHPU6mSizf1ZPqztm9RTEE/QIOh7wotAcM9CAqicbIiUMJZZxTgSnyhWF7Rz0CMxuJOiRJxgp0cy/HWxxL8i/vTVuB/61Pf/2/Nvz7w5Par+VLSY7WSR3uzHt2mc4K1Sx3Hagvblg9lDirUU5qG65f6g/k6lsLKqnSNcKDnzAK6e/Wy6deqAk8CpaJpF7TVPCAaY0qK5aS6kFA112Bz8YhR00M1RYDA45D/LVr6r89hX2euW3J99fP/k2i6w8Ltes2zNrX5FjcvFabFc2zoeMywJ67q3L04efb96s4vT94CiLqr1Td4tc5P7IxQYSZpTpzol+qhZYzwm0olIgPUokYel33zHA5Zq4ikugh0trdghctIRrIFkqJKH2xDXgjmbOo9Pvqxk6G3j3pCMX+d6k+4KRiz3lnjrl7jwqYAqWx0pu8Q5eU3DPxD3vIz3sy+oibO6WLeua69ZC8OON2Ri+E1f4QL8PNgbpm5th8ikaf+ZfDrasUtLuy9NgauUNYRa6KQ7fdo/DV63+O3aAXqOWM8PRf4c1RQxaYD/DpBREY5LYLf0ySlIfsWHsBmfEmhOMF7SKH2BZrXZ+nt0U7y2rL81z2ojTVS+rkP0QA4QkgH/ADnjld6sR8s80vAt+CmfhKAownBszfI9wcYJx+LQO1C4oZC0jgSUMjMRGqSeietNunh+sRQ9xskqx/cbG5RAK4PbBURh8S13d1tU9Zk2HubjHE6zSYiq1IRR3zcZYvsO36brVulzvn6KlS+79CbOEB8PomQGXVyGr4vv+mUWI0zyt+91z8tYjL495d5W0f7pf9+lmIwnTwID6djPnWafaMzzMSFtxc9o20nrT20n/UXFtU9BQAKtrMRKmB8y2KqxiQCU1FLRW55Xr5XSJlUgw2ASLKAq2SykSCUqu4oaeYma7ZlbvYq99OVbv7bU7lLSXPav3uN+z+uk83f+HTUCLvgJtvnEa10jG2OY/8quNM7gOhHQnJni8XIwzTot8P3lin//YrMHhuWrrFSM7GX7proZfZjsbfq0n6QHmviNCas2dtbehcKUaUK2ZYExT0Mi5pTt4WjFDteXSCq4IPxvq1q+qkPfW3h0iRXvq7qm7p+7nn26N9irEV+VYTGy3zYgVsV+C73xNckJzLdan3X+JwsUinN9mh+O5xQLjkXCspR6lr8nWAJtu4kZ6KFdX44u2CrQJb3YHoN793aOZKxHdibdtLcJ2K20RHplSwoBjSZa3qIHCldWEcyo5hirZXbRvZlxXZakg6tTV767G4mOz96sZiw05x6hY27N2z9pfj7W7LSrefh3YPA6GT2vTb8PqVca+tgR7GwnXKFvJJmarMuz3RYVV42zIQn75bETpY3hrik/slmnZvrxpWXpNy674mC/fuz8XrepaqVRI0Kk9Wfv4gHMjjbQUq6qwBn9lObBWK66ZgrZY1LSjOdkqrSUTEs/Qv3p1tL/DQibjaPbhLpzHE2AGRNkRDBlZYLUEJL5ghKkrSq6oeE/1NdOY+MJK2Anw7wi5JngEXB9yGEXDER0JIUc6mgg75poOh4JEESbZJUMb2qGNeKiUpuFYjHD7MeTjYcSpGI+oyahymYyS2Qf4ZmfQepFxKdXDK0Ddq5x706sHrKUdwbs3HiPlhU+zJAQGm4ZMKuglQPiQh3wIY5hYw+gwshGDFRtNaEiNUmMzUfBfyaRWEZMTPpJsOA4NDCsahiEIH8EKAd7f3cX4rk8ixmRobUSGMowmEy6pISIymo61NJRKOdYTpscXxdI6tg/vwn8m83/54U/Jv/7p367oQP9ufR3fyHyK0piRhhWwQkU8Ihaeo1XjkAyJIWY4hgcrVAh3cQsEQJV3y7cX/3B+iMhaq/z/8AZu3zVZ4io9At8kj/Co/hnh7hKuh58+zKPH2vah2KTAB3+7WPPcNcBWPFy4l/capiyvuXGL03hdtV9mz3SXz3Qn7defkU51+3WOC1ap2k6trX2klJC1j6TWut4xK/delcXrraQQrP6RZPVWynLcJDbsHRsqJ2ynX6HC+7Hyfqqt947SK4MSrxAKSwBYc3u/QraM0lkYvMf+8MEHx2UODV2mOU5hdQwgg4Ni4K3RFFFoQOmAfDdaUY7PHUTPU9iSwrYNcQ37DegF7LyAJxbJ/VNnPBPkWsLbyLnmohHPjgDBrXiG7+9Vtue+SqPZJNtgPIdskkgaRqGlTCsdCjocjScRURQAhMB/QooqmLXjCNNMaEJG41HIhbHhWAOwEHVGyJYtyIcC4PI/NziXf5DDXf4d2PTLcC+DvEx9cdBWd0+q2TWrZ6W58vX3stb3HEDSep2TVjjboa3p3pbv0FZ2b6q6N60PoI56bZcBAdsuA9i2XEbEbrsMuFy73KFUzfOprzp4jT97VvAVsPLL/weEl2yI"))
assert hashlib.sha256(RAW).hexdigest()==EMBEDDED_SHA256
EVIDENCE=json.loads(RAW.decode())
FAILURES=EVIDENCE['failures'];ENTROPY_REF=EVIDENCE['entropy_reference']
assert len(FAILURES)==16 and len(ENTROPY_REF)==64
assert Counter(x['dataset'] for x in FAILURES)=={'countdown_fresh':7,'gsm8k':5,'svamp':4}
assert all(x['status']=='ok' for x in ENTROPY_REF)
assert len({(x['failure_id'],x['action']) for x in ENTROPY_REF})==64
MODEL_ID=EVIDENCE['model_manifest']['model_id']
MODEL_REVISION=EVIDENCE['model_manifest']['model_revision']
assert MODEL_ID==EVIDENCE['entropy_manifest']['model_id']
assert MODEL_REVISION==EVIDENCE['entropy_manifest']['model_revision']
print('Evidence SHA',EMBEDDED_SHA256,'frozen failures',len(FAILURES),'entropy references',len(ENTROPY_REF))

In [ ]:
import ast as pyast, math, re
FINAL_RE = re.compile(r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)[ \t]*$")
NUM_RE = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")
COUNTDOWN_PROMPT_RE = re.compile(r"Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+([-+]?(?:\d+(?:\.\d*)?|\.\d+))", flags=re.I)
ARITH_RUN_RE = re.compile(r"[0-9eE+\-*/().=\s]{3,}")
ALLOWED_BINOPS = (pyast.Add, pyast.Sub, pyast.Mult, pyast.Div)
ALLOWED_UNARY = (pyast.UAdd, pyast.USub)

def close_num(a,b):
    return a is not None and math.isclose(float(a),float(b),abs_tol=1e-8,rel_tol=1e-8)

def strict_final(text):
    v=FINAL_RE.findall(str(text))
    if not v:return None
    try:
        x=float(v[-1]); return x if math.isfinite(x) else None
    except Exception:return None

def last_number(text):
    v=NUM_RE.findall(str(text).replace(',',''))
    if not v:return None
    try:
        x=float(v[-1]); return x if math.isfinite(x) else None
    except Exception:return None

def parse_countdown_prompt(prompt):
    m=COUNTDOWN_PROMPT_RE.search(str(prompt))
    if not m:raise ValueError('Countdown prompt format not recognized.')
    return [float(x) for x in NUM_RE.findall(m.group(1))],float(m.group(2))

def safe_arith(expr):
    try:tree=pyast.parse(expr,mode='eval')
    except Exception:return None
    lits=[]
    def walk(n):
        if isinstance(n,pyast.Expression):return walk(n.body)
        if isinstance(n,pyast.Constant) and isinstance(n.value,(int,float)) and not isinstance(n.value,bool):
            v=float(n.value)
            if not math.isfinite(v):raise ValueError
            lits.append(v);return v
        if isinstance(n,pyast.BinOp) and isinstance(n.op,ALLOWED_BINOPS):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,pyast.Add):return a+b
            if isinstance(n.op,pyast.Sub):return a-b
            if isinstance(n.op,pyast.Mult):return a*b
            if isinstance(n.op,pyast.Div):
                if abs(b)<1e-15:raise ZeroDivisionError
                return a/b
        if isinstance(n,pyast.UnaryOp) and isinstance(n.op,ALLOWED_UNARY):
            v=walk(n.operand);return -v if isinstance(n.op,pyast.USub) else v
        raise ValueError
    try:
        v=walk(tree)
        return (float(v),lits) if math.isfinite(v) else None
    except Exception:return None

def same_multiset(a,b):
    if len(a)!=len(b):return False
    return all(close_num(x,y) for x,y in zip(sorted(map(float,a)),sorted(map(float,b))))

def countdown_task_success(text,prompt):
    nums,target=parse_countdown_prompt(prompt)
    cleaned=(str(text).replace('×','*').replace('÷','/').replace('\\times','*').replace('\\div','/'))
    cand=[]
    for m in ARITH_RUN_RE.finditer(cleaned):
        run=m.group(0).strip()
        if any(o in run for o in ['+','-','*','/']):cand.extend(x.strip() for x in run.split('=') if x.strip())
    for expr in sorted(set(cand),key=len,reverse=True):
        p=safe_arith(expr)
        if p and same_multiset(p[1],nums) and close_num(p[0],target):return True,expr
    return False,None

def verify_repair(text,failure):
    strict,numeric=strict_final(text),last_number(text)
    strict_ok,numeric_ok=close_num(strict,failure['target']),close_num(numeric,failure['target'])
    if failure['dataset']=='countdown_fresh':task,expr=countdown_task_success(text,failure['prompt'])
    else:task,expr=bool(numeric_ok),None
    return {'strict_prediction':strict,'numeric_prediction':numeric,'target':float(failure['target']),
            'strict_success':bool(strict_ok),'numeric_success':bool(numeric_ok),'task_success':bool(task),
            'valid_countdown_expression':expr}

_demo='Use the numbers 19, 29, and 24 exactly once to produce the target 14.'
assert countdown_task_success('19 + 24 - 29 = 14',_demo)[0]
assert not countdown_task_success('29 - 19 = 10',_demo)[0]
assert verify_repair('FINAL_ANSWER: 4',{'target':4,'dataset':'gsm8k','prompt':'x'})['task_success']

In [ ]:
for f in FAILURES:assert not verify_repair(f['output'],f)['task_success'],f['failure_id']
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION)
from types import SimpleNamespace
gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
start=time.time();decoder=load_decoder(args);model=decoder.model;tokenizer=decoder.tokenizer
assert next(model.parameters()).dtype==torch.float32
print('FP32 loaded in',round(time.time()-start,1),'s, allocated',round(torch.cuda.memory_allocated()/2**30,2),'GiB')

### Mechanically audited suffix interventions

In [ ]:
ACTIONS=['open_last_25','open_last_50','open_last_75','global']
PROMPT_SUFFIX=("\n\nSolve the problem carefully. For ordinary arithmetic word problems, end with `FINAL_ANSWER: <number>`. "
               "For Countdown, include a valid arithmetic expression using every supplied number exactly once.")
Q=0.4;TAU=0.4;MODE='left1';WINDOW=16

def base_prompt_ids(f):
    return list(tokenizer.apply_chat_template([{'role':'user','content':f['prompt']+PROMPT_SUFFIX}],
                                               tokenize=True,add_generation_prompt=True,enable_thinking=False))

def response_ids(text):
    return tokenizer(text,add_special_tokens=False,return_attention_mask=False)['input_ids']

def scope_definition(n,action):
    if action=='global':return 0,n
    frac={'open_last_25':.75,'open_last_50':.50,'open_last_75':.25}[action]
    cut=max(1,min(n-1,int(math.floor(n*frac))))
    return cut,n-cut

@torch.inference_mode()
def run_repair(f,action):
    original=response_ids(f['output'])
    assert len(original)>=4
    cut,budget=scope_definition(len(original),action)
    preserved=original[:cut]
    cfg=DecodeConfig(q=Q,tau=TAU,mode=MODE,window_size=WINDOW,max_new_tokens=budget,position_penalty=None)
    cfg.validate()
    prompt=base_prompt_ids(f)+preserved
    torch.cuda.reset_peak_memory_stats()
    result=decoder.generate(prompt,cfg)
    suffix=list(result['token_ids']);full=preserved+suffix
    assert full[:cut]==preserved
    answer=tokenizer.decode(full,skip_special_tokens=True).strip()
    verify=verify_repair(answer,f)
    pred_old=last_number(f['output']);pred_new=verify['numeric_prediction']
    same_wrong=(not verify['task_success'] and pred_old is not None and pred_new is not None
                and close_num(pred_old,pred_new))
    return {'trial_id':f['failure_id']+'|left1|'+action,'failure_id':f['failure_id'],
            'problem_id':f['problem_id'],'dataset':f['dataset'],'action':action,'decoder_mode':MODE,
            'response_tokens':len(original),'preserved_tokens':cut,'requested_new_tokens':budget,
            'generated_tokens':int(result['generated_tokens']),
            'forced_fraction':cut/len(original),'exact_prefix_preserved':bool(full[:cut]==preserved),
            'token_position_preservation':sum(int(a==b) for a,b in zip(original,full))/max(len(original),len(full),1),
            'task_success':bool(verify['task_success']),'numeric_success':bool(verify['numeric_success']),
            'strict_success':bool(verify['strict_success']),'same_wrong_numeric':bool(same_wrong),
            'verification':verify,'original_output':f['output'],'repaired_output':answer,
            'outer_steps':int(result['outer_steps']),'recurrent_passes':int(result['recurrent_passes']),
            'loop_token_passes':int(result['loop_token_passes']),
            'mean_exit_depth':result['mean_exit_depth'],'wall_seconds':float(result['wall_seconds']),
            'peak_vram_gb':float(torch.cuda.max_memory_allocated()/2**30)}

smoke=run_repair(FAILURES[0],'open_last_75')
assert smoke['exact_prefix_preserved'] and smoke['outer_steps']>0
print('FP32 left1 smoke PASS', {k:smoke[k] for k in ['task_success','outer_steps','recurrent_passes','loop_token_passes','wall_seconds']})

In [ ]:
# Resumable output: separate from Notebook 17 and all previous notebooks.
SAVE_TO_DRIVE=True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT=Path('/content/drive/MyDrive/RepairScopeX/Notebook16_ALoDLM_Left1_Ablation')
else:OUT=Path('/content/RepairScopeX_Notebook16_ALoDLM_Left1_Ablation')
OUT.mkdir(parents=True,exist_ok=True)
config={'protocol':'rsx16-left1-v1','source_sha256':EMBEDDED_SHA256,'model':MODEL_ID,'revision':MODEL_REVISION,
        'code_commit':EXPECTED_CODE_COMMIT,'dtype':'fp32','mode':MODE,'q':Q,'tau':TAU,'window':WINDOW,'actions':ACTIONS}
SIGNATURE=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
MANIFEST=OUT/'run_manifest.json';LOG=OUT/'left1_repair_trials.jsonl'
if MANIFEST.exists():
    assert json.loads(MANIFEST.read_text())['run_signature']==SIGNATURE,'Different existing run. Choose a new output folder.'
else:MANIFEST.write_text(json.dumps({'run_signature':SIGNATURE,**config,'created_utc':datetime.now(timezone.utc).isoformat()},indent=2))

def load_log():
    completed={}
    if LOG.exists():
        for i,l in enumerate(LOG.read_text().splitlines(),1):
            if not l.strip():continue
            obj=json.loads(l)
            assert obj['run_signature']==SIGNATURE and obj['trial_id'] not in completed,(i,obj['trial_id'])
            completed[obj['trial_id']]=obj
    return completed

expected={f['failure_id']+'|left1|'+a for f in FAILURES for a in ACTIONS}
assert len(expected)==64
print('Already saved',len(load_log()),'/64')

In [ ]:
completed=load_log()
with LOG.open('a',encoding='utf-8') as log:
    for i,(f,action) in enumerate(((f,a) for f in FAILURES for a in ACTIONS),1):
        tid=f['failure_id']+'|left1|'+action
        if tid in completed:
            print(f'[{i}/64] SKIP',tid);continue
        try:
            rec=run_repair(f,action)
            rec.update(run_signature=SIGNATURE,status='ok',error=None)
        except Exception as exc:
            rec={'trial_id':tid,'failure_id':f['failure_id'],'action':action,'problem_id':f['problem_id'],
                 'run_signature':SIGNATURE,'status':'error','error':f'{type(exc).__name__}: {exc}'}
            torch.cuda.empty_cache();gc.collect()
        log.write(json.dumps(rec,ensure_ascii=False)+'\n');log.flush()
        print(f'[{i}/64]',tid,rec.get('task_success'),rec['status'])
print('Sweep finished; use next cell to validate.')

In [ ]:
left=list(load_log().values())
assert len(left)==64 and {x['trial_id'] for x in left}==expected
errors=[x for x in left if x['status']!='ok']
if errors:raise RuntimeError('Repair failures; see JSONL and inspect errors: '+str(errors[:3]))
assert all(x['exact_prefix_preserved'] for x in left)
L=pd.DataFrame(left).sort_values(['failure_id','action'])
R=pd.DataFrame(ENTROPY_REF).sort_values(['failure_id','action'])
L['decoder_mode']='left1';R['decoder_mode']='entropy'
assert set(zip(L.failure_id,L.action))==set(zip(R.failure_id,R.action))
assert R['exact_prefix_preserved'].all()
print('Integrity: 64 left1 vs 64 entropy; paired failure/action; preservation PASS.')

def desc(df):
    return df.groupby(['decoder_mode','action'],as_index=False).agg(
        trials=('task_success','size'),success=('task_success','mean'),
        preservation=('token_position_preservation','mean'),
        wrong_recurrence_all=('same_wrong_numeric','mean'),
        loop_passes=('loop_token_passes','mean'),time_seconds=('wall_seconds','mean'))
SUMMARY=desc(pd.concat([L,R],ignore_index=True))
display(SUMMARY)
paired=L[['failure_id','dataset','action','task_success','same_wrong_numeric','token_position_preservation','loop_token_passes','wall_seconds']].merge(
    R[['failure_id','action','task_success','same_wrong_numeric','token_position_preservation','loop_token_passes','wall_seconds']],
    on=['failure_id','action'],suffixes=('_left1','_entropy'),validate='one_to_one')
assert len(paired)==64
paired['delta_success_left1_minus_entropy']=paired['task_success_left1'].astype(int)-paired['task_success_entropy'].astype(int)
paired['delta_loop_passes_left1_minus_entropy']=paired['loop_token_passes_left1']-paired['loop_token_passes_entropy']
paired['delta_time_left1_minus_entropy']=paired['wall_seconds_left1']-paired['wall_seconds_entropy']
contrast=[]
for action,sub in paired.groupby('action'):
    only_left=int(((sub.task_success_left1==True)&(sub.task_success_entropy==False)).sum())
    only_entropy=int(((sub.task_success_left1==False)&(sub.task_success_entropy==True)).sum())
    n=only_left+only_entropy
    contrast.append({'action':action,'left1_success':float(sub.task_success_left1.mean()),
                     'entropy_success':float(sub.task_success_entropy.mean()),
                     'delta_left1_minus_entropy':float(sub.delta_success_left1_minus_entropy.mean()),
                     'only_left1':only_left,'only_entropy':only_entropy,
                     'two_sided_exact_p':float(binomtest(only_left,n,.5,alternative='two-sided').pvalue) if n else 1.,
                     'mean_delta_loop_passes':float(sub.delta_loop_passes_left1_minus_entropy.mean()),
                     'mean_delta_seconds':float(sub.delta_time_left1_minus_entropy.mean())})
CONTRAST=pd.DataFrame(contrast)
display(CONTRAST)

In [ ]:
# One output bundle, including both exact trial matrices and evidence provenance.
L.to_csv(OUT/'left1_trial_flat.csv',index=False)
SUMMARY.to_csv(OUT/'decoder_ablation_summary.csv',index=False)
paired.to_csv(OUT/'paired_decoder_mode_results.csv',index=False)
CONTRAST.to_csv(OUT/'paired_decoder_mode_tests.csv',index=False)
(OUT/'frozen_entropy_trials.jsonl').write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in ENTROPY_REF))
(OUT/'gate.json').write_text(json.dumps({
    'trials_left1':len(L),'reference_entropy_trials':len(R),
    'all_left1_passed':len(L)==64 and L.exact_prefix_preserved.all(),
    'model_precision':'fp32','decoder_modes':['left1','entropy'],'exploratory':True,
    'best_left1_action':str(SUMMARY[SUMMARY.decoder_mode=='left1'].sort_values('success',ascending=False).iloc[0].action)},indent=2))
files_out=[x for x in OUT.iterdir() if x.is_file() and x.name!='sha256_manifest.json']
(OUT/'sha256_manifest.json').write_text(json.dumps({f.name:hashlib.sha256(f.read_bytes()).hexdigest() for f in files_out},indent=2))
zip_path=shutil.make_archive('/content/RSX16_ALoDLM_Decoder_Ablation_results','zip',root_dir=OUT)
print('ZIP ready:',zip_path,Path(zip_path).stat().st_size,'bytes')
from google.colab import files
files.download(zip_path)

### Interpretive limits
- Both modes use the **same fixed 16 failures**; this is an ablation, not an independent replication.
- `left1` retries can be deterministically identical to the source for equivalent budgets. Lack of improvement would itself be informative.
- Differences in suffix/global compute must be reported; no matched-compute claim is made.
- Even with FP32, this model is architecturally different from Dream. Do **not** pool raw outcome rates across models as a causal estimate.